# SANAD — Stage 2, Experiment B
## Dense vs BM25 vs Hybrid Retrieval

### Purpose
This notebook evaluates the retrieval strategy **after the embedding experiment has already selected BGE-M3**.

It does **not** rebuild document embeddings and does **not** rebuild the FAISS index.

### Upstream dependency
The notebook uses the saved output from the previous embedding experiment:

- `bge_m3_saudi_legal.index`

### Retrieval methods compared
1. **Dense Retrieval** — BGE-M3 query embedding + the existing BGE-M3 FAISS index
2. **BM25 Retrieval** — lexical retrieval
3. **Hybrid Retrieval** — Dense + BM25 using Reciprocal Rank Fusion (RRF)

### Output for Experiment C
After the comparison, the best retrieval method is selected from the actual results and its **Top-10 candidates** are exported to:

`SANAD_Experiment_B_Top10_for_Reranking.csv`

Experiment C should use this file directly and should **not repeat retrieval**.

## 1. Install required libraries

In [1]:
!pip install -q faiss-cpu rank-bm25 sentence-transformers pyarrow openpyxl

## 2. Upload the three required files

Upload exactly:

1. `saudi_legal_moj_clean.parquet`
2. `SANAD_Retrieval_Test_Set_60.xlsx`
3. `bge_m3_saudi_legal.index`

In [2]:
from google.colab import files

uploaded = files.upload()
print("Uploaded files:")
for name in uploaded:
    print("-", name)

Saving bge_m3_saudi_legal.index to bge_m3_saudi_legal.index
Saving SANAD_Retrieval_Test_Set_60.xlsx to SANAD_Retrieval_Test_Set_60.xlsx
Saving saudi_legal_moj_clean.parquet to saudi_legal_moj_clean.parquet
Uploaded files:
- bge_m3_saudi_legal.index
- SANAD_Retrieval_Test_Set_60.xlsx
- saudi_legal_moj_clean.parquet


## 3. Load and verify the upstream files

In [3]:
import pandas as pd
import numpy as np
import faiss
import re
import time
from pathlib import Path
from tqdm.auto import tqdm

DATA_PATH  = "/content/saudi_legal_moj_clean.parquet"
TEST_PATH  = "/content/SANAD_Retrieval_Test_Set_60.xlsx"
INDEX_PATH = "/content/bge_m3_saudi_legal.index"

for p in [DATA_PATH, TEST_PATH, INDEX_PATH]:
    assert Path(p).exists(), f"Missing required file: {p}"

df = pd.read_parquet(DATA_PATH).reset_index(drop=True)

test_df = pd.read_excel(
    TEST_PATH,
    sheet_name="Test_Set",
    dtype={
        "query_id": str,
        "query_type": str,
        "query": str,
        "gold_law": str,
        "gold_article": str,
    }
).reset_index(drop=True)

bge_index = faiss.read_index(INDEX_PATH)

print("Dataset rows     :", len(df))
print("Test queries     :", len(test_df))
print("FAISS vectors    :", bge_index.ntotal)
print("Vector dimension :", bge_index.d)

required_data_cols = {"law_name", "article_number", "text"}
required_test_cols = {"query_id", "query_type", "query", "gold_law", "gold_article"}

assert required_data_cols.issubset(df.columns), "Dataset columns are incomplete."
assert required_test_cols.issubset(test_df.columns), "Test-set columns are incomplete."
assert len(df) == bge_index.ntotal, "FAISS index and dataset row count do not match."
assert len(test_df) == 60, "Expected exactly 60 test queries."

print("Verification passed ✅")

Dataset rows     : 4226
Test queries     : 60
FAISS vectors    : 4226
Vector dimension : 1024
Verification passed ✅


## 4. Build the same document representation used by the embedding experiment

This is used for BM25 only.

**Important:** Dense document embeddings are **not recalculated** here. Dense retrieval uses the existing FAISS index from the embedding experiment.

In [4]:
def build_document_text(row):
    return (
        "اسم النظام: " + str(row["law_name"]) +
        "\nرقم المادة: " + str(row["article_number"]) +
        "\nالنص القانوني: " + str(row["text"])
    )

documents = [build_document_text(row) for _, row in df.iterrows()]

print("Documents prepared for BM25:", len(documents))
print("\nExample:\n")
print(documents[0][:700])

Documents prepared for BM25: 4226

Example:

اسم النظام: نظام مكافحة جرائم الإرهاب وتمويله
رقم المادة: المادة الأولى
النص القانوني: المادة الأولى
يقصد بالألفاظ والعبارات الآتية -أينما وردت في هذا النظام- المعاني الموضحة أمام كل منهاء ما لم يقتض السياق خلافذلك:
-١ النظام: نظام مكافحة جرائم الإرهابوتمويله.
-١ اللائحة: اللائحة التنفيذيةللنظام.
؟- الجريمة الإرهابية: كل سلوك يقوم به الجاني تنفيذاً لمشروع إجرامي فردي أو جماعي بشكل مباشر أو غير مباشرء يقصد به الإخلال بالنظام العام» أو زعزعة أمنالمجتمع
واستقرار الدولة أو تعريض وحدتها الوطنية للخطرء أو تعطيل النظام الأساسي للحكم أو بعض أحكامه؛ أو إلحاق الضرر بأحد مرافق الدولة أو مواردها الطبيعية أو الاقتصادية»أو
محاولة إرغام إحدى سلطاتها على القيام بعمل ما أو الامتناع عنه» أو إيذاء أي شخص أو ال


## 5. Arabic normalization and BM25 tokenization

In [5]:
ARABIC_DIACRITICS = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0640]")

def normalize_arabic(text):
    text = str(text)
    text = re.sub(ARABIC_DIACRITICS, "", text)
    text = re.sub(r"[إأآٱ]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_arabic(text):
    return normalize_arabic(text).split()

tokenized_documents = [tokenize_arabic(doc) for doc in documents]
print("BM25 tokenization ready ✅")

BM25 tokenization ready ✅


## 6. Build BM25

In [6]:
from rank_bm25 import BM25Okapi

BM25_K1 = 1.5
BM25_B  = 0.75

bm25 = BM25Okapi(tokenized_documents, k1=BM25_K1, b=BM25_B)

print("BM25 built ✅")
print("k1 =", BM25_K1)
print("b  =", BM25_B)

BM25 built ✅
k1 = 1.5
b  = 0.75


## 7. Load BGE-M3 for query embeddings only

The legal-document embeddings are already stored in `bge_m3_saudi_legal.index`.

BGE-M3 is loaded here only to convert each new query into the same embedding space.

In [7]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

bge_model = SentenceTransformer("BAAI/bge-m3", device=device)
bge_model.max_seq_length = 2048

print("BGE-M3 query encoder loaded ✅")
print("Device:", device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (8 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

BGE-M3 query encoder loaded ✅
Device: cuda


## 8. Gold-label consistency check

The same whitespace-insensitive matching rule must later be used by Experiment C.

This cell does **not** change any gold label. If the test set and the knowledge base do not match, the experiment stops instead of producing misleading metrics.

In [8]:
from collections import Counter

def norm_key(x):
    return re.sub(r"\s+", "", str(x))

df_law_norm = df["law_name"].map(norm_key).to_numpy()
df_article_norm = df["article_number"].map(norm_key).to_numpy()

raw_keys = set(
    zip(
        df["law_name"].astype(str).str.strip(),
        df["article_number"].astype(str).str.strip()
    )
)
normalized_keys = set(zip(df_law_norm, df_article_norm))

assert len(normalized_keys) == len(raw_keys), (
    "Whitespace normalization merged distinct legal records."
)

key_counts = Counter(zip(df_law_norm, df_article_norm))
missing = []
duplicates = []

for _, row in test_df.iterrows():
    key = (norm_key(row["gold_law"]), norm_key(row["gold_article"]))
    count = key_counts[key]

    if count == 0:
        missing.append({
            "query_id": row["query_id"],
            "gold_law": row["gold_law"],
            "gold_article": row["gold_article"],
        })
    elif count > 1:
        duplicates.append({
            "query_id": row["query_id"],
            "count": count,
            "gold_law": row["gold_law"],
            "gold_article": row["gold_article"],
        })

print("Normalization safe ✅")
print("Gold answers missing from data:", len(missing))
print("Duplicate gold keys:", len(duplicates))

if missing:
    print("\nMissing gold labels:")
    for item in missing:
        print(item["query_id"], "|", item["gold_law"], "|", item["gold_article"])

if duplicates:
    print("\nDuplicate gold labels:")
    for item in duplicates:
        print(item)

assert len(missing) == 0, (
    "STOP: Some test-set gold labels do not exist in the knowledge base. "
    "Fix this consistency issue before calculating retrieval metrics."
)

assert len(duplicates) == 0, (
    "STOP: Some gold law/article keys map to more than one record."
)

print("\nAll 60 gold answers are valid ✅")

Normalization safe ✅
Gold answers missing from data: 0
Duplicate gold keys: 0

All 60 gold answers are valid ✅


## 9. Retrieval functions

In [9]:
RRF_K = 60

def dense_ranking(query):
    query_embedding = bge_model.encode(
        [str(query)],
        normalize_embeddings=True
    )
    query_embedding = np.asarray(query_embedding, dtype="float32")

    scores, indices = bge_index.search(query_embedding, bge_index.ntotal)

    indices = indices[0].astype(int)
    scores = scores[0].astype(float)

    assert (indices >= 0).all(), "FAISS returned an invalid index."
    assert len(np.unique(indices)) == bge_index.ntotal, "Dense ranking is incomplete."

    return indices, scores

def bm25_ranking(query):
    tokens = tokenize_arabic(query)
    raw_scores = np.asarray(bm25.get_scores(tokens), dtype=np.float64)

    indices = np.argsort(-raw_scores, kind="stable")
    scores = raw_scores[indices]

    return indices.astype(int), scores.astype(float)

def reciprocal_rank_fusion(dense_indices, bm25_indices, k=60):
    n = len(df)

    dense_rank = np.empty(n, dtype=np.int32)
    bm25_rank = np.empty(n, dtype=np.int32)

    dense_rank[dense_indices] = np.arange(1, n + 1)
    bm25_rank[bm25_indices] = np.arange(1, n + 1)

    rrf_scores = (
        1.0 / (k + dense_rank) +
        1.0 / (k + bm25_rank)
    )

    hybrid_indices = np.argsort(-rrf_scores, kind="stable")
    hybrid_scores = rrf_scores[hybrid_indices]

    return hybrid_indices.astype(int), hybrid_scores.astype(float)

def get_gold_rank(ranking_indices, gold_law, gold_article):
    gold_law = norm_key(gold_law)
    gold_article = norm_key(gold_article)

    for rank, idx in enumerate(ranking_indices, start=1):
        if (
            df_law_norm[int(idx)] == gold_law
            and df_article_norm[int(idx)] == gold_article
        ):
            return rank

    return None

def metric_values(gold_rank):
    return {
        "Recall@3": int(gold_rank is not None and gold_rank <= 3),
        "Recall@5": int(gold_rank is not None and gold_rank <= 5),
        "Recall@10": int(gold_rank is not None and gold_rank <= 10),
        "RR": 0.0 if gold_rank is None else 1.0 / gold_rank,
    }

print("Retrieval functions ready ✅")

Retrieval functions ready ✅


## 10. Warm-up

In [10]:
_ = bge_model.encode(["ما حقوق العامل؟"], normalize_embeddings=True)
_ = bm25.get_scores(tokenize_arabic("ما حقوق العامل؟"))
print("Warm-up completed ✅")

Warm-up completed ✅


## 11. Evaluate Dense, BM25, and Hybrid on the same 60 queries

In [11]:
rows = []
ranking_cache = {}

for _, test_row in tqdm(test_df.iterrows(), total=len(test_df), desc="Experiment B"):
    query_id = str(test_row["query_id"])
    query_type = str(test_row["query_type"])
    query = str(test_row["query"])
    gold_law = str(test_row["gold_law"]).strip()
    gold_article = str(test_row["gold_article"]).strip()

    start = time.perf_counter()
    dense_indices, dense_scores = dense_ranking(query)
    dense_time = time.perf_counter() - start

    start = time.perf_counter()
    bm25_indices, bm25_scores = bm25_ranking(query)
    bm25_time = time.perf_counter() - start

    start = time.perf_counter()
    hybrid_indices, hybrid_scores = reciprocal_rank_fusion(
        dense_indices, bm25_indices, k=RRF_K
    )
    fusion_only_time = time.perf_counter() - start
    hybrid_time = dense_time + bm25_time + fusion_only_time

    rankings = {
        "Dense": (dense_indices, dense_scores),
        "BM25": (bm25_indices, bm25_scores),
        "Hybrid": (hybrid_indices, hybrid_scores),
    }
    ranking_cache[query_id] = rankings

    result = {
        "query_id": query_id,
        "query_type": query_type,
        "query": query,
        "gold_law": gold_law,
        "gold_article": gold_article,
    }

    method_times = {
        "Dense": dense_time,
        "BM25": bm25_time,
        "Hybrid": hybrid_time,
    }

    for method, (indices, scores) in rankings.items():
        gold_rank = get_gold_rank(indices, gold_law, gold_article)
        metrics = metric_values(gold_rank)

        result[f"{method}_Gold_Rank"] = gold_rank
        result[f"{method}_Recall@3"] = metrics["Recall@3"]
        result[f"{method}_Recall@5"] = metrics["Recall@5"]
        result[f"{method}_Recall@10"] = metrics["Recall@10"]
        result[f"{method}_RR"] = metrics["RR"]
        result[f"{method}_Time_s"] = method_times[method]

        for rank in range(10):
            idx = int(indices[rank])
            legal_row = df.iloc[idx]

            result[f"{method}_Top{rank+1}_DocIndex"] = idx
            result[f"{method}_Top{rank+1}_Law"] = str(legal_row["law_name"])
            result[f"{method}_Top{rank+1}_Article"] = str(legal_row["article_number"])
            result[f"{method}_Top{rank+1}_Score"] = float(scores[rank])

    rows.append(result)

results_df = pd.DataFrame(rows)

print("Finished ✅")
print("Queries evaluated:", len(results_df))
display(results_df.head())

Experiment B:   0%|          | 0/60 [00:00<?, ?it/s]

Finished ✅
Queries evaluated: 60


,query_id,query_type,query,gold_law,gold_article,Dense_Gold_Rank,Dense_Recall@3,Dense_Recall@5,Dense_Recall@10,Dense_RR,...,Hybrid_Top8_Article,Hybrid_Top8_Score,Hybrid_Top9_DocIndex,Hybrid_Top9_Law,Hybrid_Top9_Article,Hybrid_Top9_Score,Hybrid_Top10_DocIndex,Hybrid_Top10_Law,Hybrid_Top10_Article,Hybrid_Top10_Score
0,Q001,formal_direct,ما الدعاوى والطلبات التي لا تسري عليها أحكام ا...,نظام التكاليف القضائية,المادة الثانية,3,1,1,1,0.333333,...,المادة الثانية والخمسون,0.024490,2828,نظام التكاليف القضائية,المادة الثالثة,0.023026,696,اللائحة التنفيذية لنظام المحاكم التجارية,المادة الخامسة عشرة بعد المائتين,0.022529
1,Q002,formal_paraphrase,ما الحالات المستثناة من تطبيق نظام التكاليف ال...,نظام التكاليف القضائية,المادة الثانية,5,0,1,1,0.200000,...,المادة العاشرة,0.025582,1973,نظام التكاليف القضائية,المادة الرابعة,0.025479,2816,نظام التكاليف القضائية,المادة الخامسةعشرة,0.025429
2,Q003,informal,وش القضايا اللي ما عليها تكاليف قضائية؟,نظام التكاليف القضائية,المادة الثانية,9,0,0,1,0.111111,...,المادة الرابعة,0.024869,3354,نظام التكاليف القضائية,المادة الخامسة,0.024625,3183,نظام الأحداث,المادة الثالثةعشرة,0.022754
3,Q004,formal_direct,ما الحد الأعلى للتكاليف القضائية المفروضة على ...,نظام التكاليف القضائية,المادة الثالثة,1,1,1,1,1.000000,...,المادة السادسة,0.026430,29,قواعد العمل في مكاتب المصالحة وإجراءاته,المادة السادسة والعشرون,0.025774,153,اللائحة التنفيذية لنظام التكاليف القضائية,المادة الثانية,0.025253
4,Q005,formal_paraphrase,كيف يتم تحديد قيمة التكاليف القضائية بالنسبة إ...,نظام التكاليف القضائية,المادة الثالثة,2,1,1,1,0.500000,...,المادة السابعة,0.027501,2241,نظام التكاليف القضائية,المادة السادسة,0.026944,4171,اللائحة التنفيذية لنظام التكاليف القضائية,المادة الثانيةعشرة,0.026398


## 12. Overall results

In [12]:
def summarize_method(method):
    return {
        "Method": method,
        "Hits@3": int(results_df[f"{method}_Recall@3"].sum()),
        "Recall@3": results_df[f"{method}_Recall@3"].mean(),
        "Hits@5": int(results_df[f"{method}_Recall@5"].sum()),
        "Recall@5": results_df[f"{method}_Recall@5"].mean(),
        "Hits@10": int(results_df[f"{method}_Recall@10"].sum()),
        "Recall@10": results_df[f"{method}_Recall@10"].mean(),
        "MRR": results_df[f"{method}_RR"].mean(),
        "Avg_Time_s": results_df[f"{method}_Time_s"].mean(),
    }

overall_df = pd.DataFrame([
    summarize_method("Dense"),
    summarize_method("BM25"),
    summarize_method("Hybrid"),
])

display(overall_df.round(4))

,Method,Hits@3,Recall@3,Hits@5,Recall@5,Hits@10,Recall@10,MRR,Avg_Time_s
0,Dense,50,0.8333,55,0.9167,58,0.9667,0.7355,0.0313
1,BM25,27,0.4500,32,0.5333,35,0.5833,0.4158,0.0120
2,Hybrid,41,0.6833,43,0.7167,46,0.7667,0.5818,0.0438


## 13. Results by query type

In [13]:
type_rows = []

for query_type, group in results_df.groupby("query_type"):
    for method in ["Dense", "BM25", "Hybrid"]:
        type_rows.append({
            "query_type": query_type,
            "Method": method,
            "Recall@3": group[f"{method}_Recall@3"].mean(),
            "Recall@5": group[f"{method}_Recall@5"].mean(),
            "Recall@10": group[f"{method}_Recall@10"].mean(),
            "MRR": group[f"{method}_RR"].mean(),
            "Avg_Time_s": group[f"{method}_Time_s"].mean(),
        })

by_type_df = pd.DataFrame(type_rows)
display(by_type_df.round(4))

,query_type,Method,Recall@3,Recall@5,Recall@10,MRR,Avg_Time_s
0,formal_direct,Dense,1.00,1.00,1.00,0.8333,0.0313
1,formal_direct,BM25,0.70,0.80,0.80,0.6438,0.0123
2,formal_direct,Hybrid,0.85,0.90,0.90,0.7492,0.0440
3,formal_paraphrase,Dense,0.80,0.90,0.95,0.6810,0.0336
4,formal_paraphrase,BM25,0.45,0.60,0.70,0.4207,0.0129
5,formal_paraphrase,Hybrid,0.75,0.80,0.80,0.6093,0.0470
6,informal,Dense,0.70,0.85,0.95,0.6920,0.0290
7,informal,BM25,0.20,0.20,0.25,0.1829,0.0108
8,informal,Hybrid,0.45,0.45,0.60,0.3868,0.0403


## 14. Paired bootstrap confidence intervals

In [14]:
rng = np.random.default_rng(42)
N_BOOT = 5000

def paired_bootstrap_diff(before, after):
    before = np.asarray(before, dtype=float)
    after = np.asarray(after, dtype=float)
    n = len(before)

    observed = after.mean() - before.mean()
    diffs = np.empty(N_BOOT)

    for i in range(N_BOOT):
        sample_idx = rng.integers(0, n, size=n)
        diffs[i] = after[sample_idx].mean() - before[sample_idx].mean()

    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return observed, lo, hi

ci_rows = []

for comparison_method in ["BM25", "Hybrid"]:
    for metric_name, dense_col, other_col in [
        ("Recall@3", "Dense_Recall@3", f"{comparison_method}_Recall@3"),
        ("Recall@5", "Dense_Recall@5", f"{comparison_method}_Recall@5"),
        ("MRR", "Dense_RR", f"{comparison_method}_RR"),
    ]:
        diff, lo, hi = paired_bootstrap_diff(
            results_df[dense_col],
            results_df[other_col]
        )

        ci_rows.append({
            "Comparison": f"{comparison_method} - Dense",
            "Metric": metric_name,
            "Difference": diff,
            "95% CI Lower": lo,
            "95% CI Upper": hi,
        })

ci_df = pd.DataFrame(ci_rows)
display(ci_df.round(4))

,Comparison,Metric,Difference,95% CI Lower,95% CI Upper
0,BM25 - Dense,Recall@3,-0.3833,-0.5333,-0.2333
1,BM25 - Dense,Recall@5,-0.3833,-0.5333,-0.2333
2,BM25 - Dense,MRR,-0.3197,-0.4516,-0.1830
3,Hybrid - Dense,Recall@3,-0.1500,-0.2833,-0.0167
4,Hybrid - Dense,Recall@5,-0.2000,-0.3167,-0.0833
5,Hybrid - Dense,MRR,-0.1537,-0.2653,-0.0419


## 15. Select the retrieval method for Experiment C

Selection priority:
1. Recall@3
2. Recall@5
3. MRR
4. Lower average retrieval time

The winner is selected from the actual run and is not hard-coded.

In [15]:
selection_df = overall_df.sort_values(
    by=["Recall@3", "Recall@5", "MRR", "Avg_Time_s"],
    ascending=[False, False, False, True],
    kind="stable"
).reset_index(drop=True)

SELECTED_RETRIEVAL = str(selection_df.loc[0, "Method"])

print("Selected retrieval method:", SELECTED_RETRIEVAL)
display(
    selection_df[
        ["Method", "Recall@3", "Recall@5", "Recall@10", "MRR", "Avg_Time_s"]
    ].round(4)
)

Selected retrieval method: Dense


,Method,Recall@3,Recall@5,Recall@10,MRR,Avg_Time_s
0,Dense,0.8333,0.9167,0.9667,0.7355,0.0313
1,Hybrid,0.6833,0.7167,0.7667,0.5818,0.0438
2,BM25,0.4500,0.5333,0.5833,0.4158,0.0120


## 16. Export the selected Top-10 candidates for Experiment C

This file is the direct handoff from B to C.

Experiment C should load this file and rerank these candidates. It should not run FAISS retrieval again.

In [16]:
top10_rows = []

for _, test_row in test_df.iterrows():
    query_id = str(test_row["query_id"])
    query = str(test_row["query"])
    query_type = str(test_row["query_type"])
    gold_law = str(test_row["gold_law"]).strip()
    gold_article = str(test_row["gold_article"]).strip()

    indices, scores = ranking_cache[query_id][SELECTED_RETRIEVAL]

    for rank in range(1, 11):
        idx = int(indices[rank - 1])
        legal_row = df.iloc[idx]

        top10_rows.append({
            "query_id": query_id,
            "query_type": query_type,
            "query": query,
            "gold_law": gold_law,
            "gold_article": gold_article,
            "selected_retrieval": SELECTED_RETRIEVAL,
            "candidate_rank": rank,
            "doc_index": idx,
            "retrieval_score": float(scores[rank - 1]),
            "candidate_law": str(legal_row["law_name"]),
            "candidate_article": str(legal_row["article_number"]),
            "candidate_text": str(legal_row["text"]),
        })

top10_df = pd.DataFrame(top10_rows)

assert len(top10_df) == 600
assert top10_df.groupby("query_id").size().eq(10).all()
assert top10_df["selected_retrieval"].nunique() == 1

TOP10_PATH = "/content/SANAD_Experiment_B_Top10_for_Reranking.csv"

top10_df.to_csv(TOP10_PATH, index=False, encoding="utf-8-sig")

print("Top-10 handoff file created ✅")
print("Selected retrieval:", SELECTED_RETRIEVAL)
print("Rows:", len(top10_df))
print("Queries:", top10_df["query_id"].nunique())
print("File:", TOP10_PATH)

display(top10_df.head(12))

Top-10 handoff file created ✅
Selected retrieval: Dense
Rows: 600
Queries: 60
File: /content/SANAD_Experiment_B_Top10_for_Reranking.csv


,query_id,query_type,query,gold_law,gold_article,selected_retrieval,candidate_rank,doc_index,retrieval_score,candidate_law,candidate_article,candidate_text
0,Q001,formal_direct,ما الدعاوى والطلبات التي لا تسري عليها أحكام ا...,نظام التكاليف القضائية,المادة الثانية,Dense,1,967,0.674757,نظام التكاليف القضائية,المادة السابعةعشرة,المادة السابعةعشرة\nمع مراعاة ما تقضي به الأنظ...
1,Q001,formal_direct,ما الدعاوى والطلبات التي لا تسري عليها أحكام ا...,نظام التكاليف القضائية,المادة الثانية,Dense,2,2305,0.663706,نظام التكاليف القضائية,المادة العاشرة,المادة العاشرة\nلاتفرض تكاليف جديدة على الدعوى...
2,Q001,formal_direct,ما الدعاوى والطلبات التي لا تسري عليها أحكام ا...,نظام التكاليف القضائية,المادة الثانية,Dense,3,1078,0.654441,نظام التكاليف القضائية,المادة الثانية,المادة الثانية\nتسري أحكام النظام على جميع الد...
3,Q001,formal_direct,ما الدعاوى والطلبات التي لا تسري عليها أحكام ا...,نظام التكاليف القضائية,المادة الثانية,Dense,4,816,0.621805,نظام التكاليف القضائية,المادة الثامنة,المادة الثامنة\n١.تُقرض على الطلبات الآتية تكا...
4,Q001,formal_direct,ما الدعاوى والطلبات التي لا تسري عليها أحكام ا...,نظام التكاليف القضائية,المادة الثانية,Dense,5,371,0.610724,نظام التكاليف القضائية,المادة السادسةعشرة,المادة السادسةعشرة\nترد التكاليف القضائية المد...
5,Q001,formal_direct,ما الدعاوى والطلبات التي لا تسري عليها أحكام ا...,نظام التكاليف القضائية,المادة الثانية,Dense,6,1088,0.610643,نظام التكاليف القضائية,المادة الثانيةعشرة,المادة الثانيةعشرة\n١.فيما عدا طلب النقض وطلب ...
6,Q001,formal_direct,ما الدعاوى والطلبات التي لا تسري عليها أحكام ا...,نظام التكاليف القضائية,المادة الثانية,Dense,7,474,0.591952,اللائحة التنفيذية لنظام المحاكم التجارية,المادة الحادية والخمسون,المادة الحادية والخمسون : الدعاوى والطلبات الت...
7,Q001,formal_direct,ما الدعاوى والطلبات التي لا تسري عليها أحكام ا...,نظام التكاليف القضائية,المادة الثانية,Dense,8,970,0.587557,اللائحة التنفيذية لنظام التكاليف القضائية,المادة الثالثة,المادة الثالثة\nتقدّر التكاليف القضاتية في الد...
8,Q001,formal_direct,ما الدعاوى والطلبات التي لا تسري عليها أحكام ا...,نظام التكاليف القضائية,المادة الثانية,Dense,9,2241,0.585913,نظام التكاليف القضائية,المادة السادسة,المادة السادسة\nدون إخلال بما ورد في المادتين ...
9,Q001,formal_direct,ما الدعاوى والطلبات التي لا تسري عليها أحكام ا...,نظام التكاليف القضائية,المادة الثانية,Dense,10,732,0.575659,نظام التكاليف القضائية,المادة الأولى,المادة الأولى\nيكون للألفاظ والعبارات الواردة ...


## 17. Save all Experiment B results

In [17]:
OUTPUT_XLSX = "/content/SANAD_Experiment_B_Retrieval_Comparison.xlsx"
OUTPUT_PER_QUERY = "/content/SANAD_Experiment_B_Per_Query.csv"

results_df.to_csv(
    OUTPUT_PER_QUERY,
    index=False,
    encoding="utf-8-sig"
)

settings_df = pd.DataFrame({
    "Setting": [
        "Upstream embedding model",
        "Upstream vector index",
        "Document embeddings rebuilt in B?",
        "Query embedding model",
        "Dense search",
        "BM25",
        "Hybrid fusion",
        "RRF k",
        "Gold matching",
        "Test queries",
        "Experiment C handoff",
    ],
    "Value": [
        "BAAI/bge-m3",
        "bge_m3_saudi_legal.index",
        "No",
        "BAAI/bge-m3",
        "Existing FAISS index",
        f"BM25Okapi (k1={BM25_K1}, b={BM25_B})",
        "Dense + BM25 Reciprocal Rank Fusion",
        RRF_K,
        "Whitespace-insensitive law name + article number",
        len(test_df),
        f"Top-10 from selected method: {SELECTED_RETRIEVAL}",
    ],
})

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    settings_df.to_excel(writer, sheet_name="Settings", index=False)
    overall_df.to_excel(writer, sheet_name="Overall", index=False)
    by_type_df.to_excel(writer, sheet_name="By_Query_Type", index=False)
    ci_df.to_excel(writer, sheet_name="Bootstrap_CI", index=False)
    results_df.to_excel(writer, sheet_name="Per_Query", index=False)
    top10_df.to_excel(writer, sheet_name="Top10_for_C", index=False)

print("Saved ✅")
print(OUTPUT_XLSX)
print(OUTPUT_PER_QUERY)
print(TOP10_PATH)

Saved ✅
/content/SANAD_Experiment_B_Retrieval_Comparison.xlsx
/content/SANAD_Experiment_B_Per_Query.csv
/content/SANAD_Experiment_B_Top10_for_Reranking.csv


## 18. Download Experiment B outputs

In [18]:
from google.colab import files

files.download("/content/SANAD_Experiment_B_Retrieval_Comparison.xlsx")
files.download("/content/SANAD_Experiment_B_Per_Query.csv")
files.download("/content/SANAD_Experiment_B_Top10_for_Reranking.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Final Experiment B flow

**Previous Embedding Experiment**  
BGE-M3 document embeddings → saved `bge_m3_saudi_legal.index`

**Experiment B**  
Existing BGE-M3 FAISS index + BGE-M3 query embedding  
vs. BM25  
vs. Hybrid RRF  
→ compare retrieval quality  
→ select the best method from actual results  
→ export its Top-10 candidates

**Experiment C must start from `SANAD_Experiment_B_Top10_for_Reranking.csv`.**